In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from bingX import BingX
import time
from bingX.spot import types

    
engulfing_body_condition = 2
engulfing_first_candle_color_dont_care = True
threesoldiers_body_condition = [0.5,0.5]


secret_key = "WqP5txrWtmbGcMND5GjFiBZICth5nZFIXYKEPqKitRl7xOHVGOFzNo8yNCLQUmK4guPTNgV4nwKLT0aD5iYIg"
api_key = "SbyekABHMR0SyrIQKTqU14sExh5m40sV5a6jaBenI26u5SDm8z4gGzQCLamCp6Ww2uJyVjyhJQESk4Gt5jA"


ModuleNotFoundError: No module named 'bingX'

## functions

In [4]:
from enum import Enum

class Strategy(Enum):
    THREESOLDIERS = 4
    ENGULFING = 3
    
class Interval(Enum):
    HOURLY_ONEHOUR = '1h'
    HOURLY_FOURHOUR = '4h'
    DAILY_ONEDAY = '1d'
    MINUTELY_ONEMINUTE = '1m'
    
class Audit(Enum):
    WEEKLY = 1
    MONTHLY = 2

def updateTotalBalance(i):
    global usdt_balance, bitcoin_balance, total_balance
    total_balance = usdt_balance + bitcoin_balance * cdf.loc[i]["close"]

def buyBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global buyDuplicateChecker,sellDuplicateChecker, learning_rate, buy_percentage_current
    global buy_percentage_init, sell_percentage_current, sell_percentage_init
   
    buyDuplicateChecker += 1
    sellDuplicateChecker = 0
    if strategy.name == Strategy.ENGULFING.name:
        if buyDuplicateChecker != 3:
            return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[92m \033[1m{}\033[00m" .format(f"buy opportunity!\n"))        

    if usdt_balance > 0:
        buyPositionsNumber += 1
        
        print(f'current buy percentage: {buy_percentage_current}')
        amountToBuy_usdt = total_balance * buy_percentage_current / 100
        buy_percentage_current *= learning_rate
        sell_percentage_current = sell_percentage_init
        
        if amountToBuy_usdt > usdt_balance:
            amountToBuy_usdt = usdt_balance
            
        addedBitcoinAmount = np.longdouble(amountToBuy_usdt / cdf.loc[i]["close"])
        usdt_balance -= amountToBuy_usdt
        bitcoin_balance += addedBitcoinAmount
        updateTotalBalance(i)
        print(f'\tbought {amountToBuy_usdt} $ BTC ({addedBitcoinAmount} BTCs) in {time.ctime(cdf.loc[i]["time"] // 1e3)}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')
        
    else:
        print(f"\tthere isn't any usdt to buy BTC with it!")
        print(f'\tcurrent BTC balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')

def sellBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global sellDuplicateChecker,buyDuplicateChecker, buy_percentage_current, buy_percentage_init
    global sell_percentage_current, sell_percentage_init, learning_rate
    
    sellDuplicateChecker += 1
    buyDuplicateChecker = 0
    
    if strategy.name == Strategy.ENGULFING.name:
        if sellDuplicateChecker != 3:
            return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[91m \033[1m{}\033[00m" .format(f"sell opportunity!\n"))        
    if bitcoin_balance > 0:
        sellPositionsNumber += 1
        
        currentValueOfAllBitcoins = bitcoin_balance * cdf.loc[i]["close"]
        
        print(f'current sell percentage: {sell_percentage_current}')
        amountToSell_usdt = total_balance * sell_percentage_current / 100
        sell_percentage_current *= learning_rate
        buy_percentage_current = buy_percentage_init
        
        if amountToSell_usdt > currentValueOfAllBitcoins:
            amountToSell_usdt = currentValueOfAllBitcoins
        
        amountToSell_bitcoin = amountToSell_usdt / cdf.loc[i]["close"]
        usdt_balance += amountToSell_usdt
        bitcoin_balance -= amountToSell_bitcoin
        updateTotalBalance(i)
        print(f'\tsold {amountToSell_usdt} $ of BTC ({amountToSell_usdt} $) in {time.ctime(cdf.loc[i]["time"] // 1e3)}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')
        
    else:
        print(f"\tthere isn't any bitcoin to sell!")
        print(f'\tcurrent USDT balance: {usdt_balance} $')

def checkForPosition(i,strategy):
    if strategy == Strategy.THREESOLDIERS:
        if i >= len(cdf) - 2:
            return
        suitableForBuy = False
        suitableForSell = False
        print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 1.0:
                if cdf.loc[2 + i]['isGreen'] == 1.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 0.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                if cdf.loc[2 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForSell = True
                
    if strategy == Strategy.ENGULFING:
        if i >= len(cdf) - 1:
            return
        suitableForBuy = False
        suitableForSell = False
        print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                        suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                if cdf.loc[2 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                        suitableForSell = True
                

        if suitableForBuy:
    #         print(f'{i} is suitable for buy!')
            buyBTC(i)

        if suitableForSell:
    #         print(f"{i} is suitable for sell!")
            sellBTC(i)

    #     if not (suitableForBuy or suitableForSell):
    #         print(f'{i} is not suitable for buy nor sell!')

def timeout(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber, counter
    updateTotalBalance(i)
    print("\033[91m \033[1m{}\033[00m" .format(f"\n\t\tCOUNTER REACHER IT'S MAXIMUM!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\topened {buyPositionsNumber} buy\
positions and {sellPositionsNumber} sell positions!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent usdt balance: {usdt_balance}\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]['Close']} $)\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent total balance: {total_balance}\n"))
    counter = 0
    buyPositionsNumber = 0
    sellPositionsNumber = 0
    return total_balance

def calculatePeriod():
    global audit, interval, period
    if audit == Audit.MONTHLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 30
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 30
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 30
            
    if audit == Audit.WEEKLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 7
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 7
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 7



## end functions

In [5]:
def get_account_info(bingx_client) -> float | np.longdouble:
    res = bingx_client.spot.trade.get_assets()
    for record in res['balances']:
        if record['asset'] == 'USDT':
            usdt_balance = float(record['free'])
        if record['asset'] == 'BTC':
            bitcoin_balance = float(record['free'])

    print(f'usdt balance: {usdt_balance}\n')
    print(f'bitcoin balance: {bitcoin_balance}')
    return usdt_balance, bitcoin_balance

In [6]:
bingx_client = BingX(api_key=api_key, secret_key=secret_key)
usdt_balance, bitcoin_balance = get_account_info(bingx_client)


NameError: name 'BingX' is not defined

In [7]:
pair = "BTC-USDT"
interval = Interval.HOURLY_ONEHOUR
strategy = Strategy.ENGULFING
audit = Audit.WEEKLY

communication_max_tries = 5

buy_percentage_init = 10
sell_percentage_init = 13
learning_rate = 1.3

total_balance = 0 # usdt_balance + bitcoin_balance * bitcoin_price
buyPositionsNumber = 0
sellPositionsNumber = 0
buyDuplicateChecker = 0
sellDuplicateChecker = 0
period = 0
buy_percentage_current = buy_percentage_init
sell_percentage_current = sell_percentage_init
usdt_balance_init = usdt_balance


print(f'chosen strategy: {strategy.name}')

NameError: name 'usdt_balance' is not defined

In [31]:
def update_balance_info(bingx_client,pair,interval,usdt_balance,bitcoin_balance: np.longdouble) -> float | float:
    res = bingx_client.perpetual_v2.market.get_k_line_data(pair,interval)
    print(f'bitcoin balance: {bitcoin_balance}')
    bitcoin_balance_as_usdt = bitcoin_balance * float(res['close'])
    print(f'bitcoin_balance_as_usdt: {bitcoin_balance_as_usdt}')
    total_balance = usdt_balance + bitcoin_balance_as_usdt
    print(f'total balance: {total_balance}')
    return bitcoin_balance_as_usdt, total_balance

In [32]:
bitcoin_balance_as_usdt, total_balance = update_balance_info(
    bingx_client,pair,interval.value,usdt_balance,bitcoin_balance)

bitcoin balance: 0.00041958
bitcoin_balance_as_usdt: 27.766839366
total balance: 89.9153698268547


In [33]:
def get_times(strategy,interval):
    if interval == '1h':
        timeOffset = 60 * 60 * 1000
    if interval == '4h':
        timeOffset = 4 * 60 * 60 * 1000
    if interval == '1d':
        timeOffset = 24 * 60 * 60 * 1000
    if interval == '1m':
        timeOffset = 60 * 1000

    start_time = round(time.time() * 1e3) - timeOffset * (strategy)
    end_time = round(time.time() * 1e3)
    print(f'difference(ms): {end_time - start_time}')
    print(f'difference(hour): {(end_time - start_time) / 1e3 / 60 / 60}')
    print(f'number of candles: {(end_time - start_time) / timeOffset - 1}')
    print(f'\tcurrent interval -> {interval}')
    return start_time, end_time,timeOffset

In [34]:
start_time, end_time, intervalAsMillisecond = get_times(strategy.value,interval.value)

difference(ms): 10800000
difference(hour): 3.0
number of candles: 2.0
	current interval -> 1h


In [35]:
klineDataHistory = bingx_client.perpetual_v2.market.get_k_line_data(pair,interval.value,start_time,end_time)
klineDataHistory

[{'open': '65354.0',
  'close': '65869.9',
  'high': '66088.0',
  'low': '65346.5',
  'volume': '3323.76',
  'time': 1711303200000},
 {'open': '65867.8',
  'close': '65856.8',
  'high': '66069.4',
  'low': '65685.9',
  'volume': '2609.91',
  'time': 1711306800000}]

In [36]:
import warnings
warnings.filterwarnings('ignore')

df = pd.DataFrame(klineDataHistory)
cdf = df.iloc[::-1]

cdf['open'] = pd.to_numeric(cdf['open'])
cdf['high'] = pd.to_numeric(cdf['high'])
cdf['low'] = pd.to_numeric(cdf['low'])
cdf['close'] = pd.to_numeric(cdf['close'])

cdf['isGreen'] = cdf["close"]
cdf['bodyLen'] = abs(cdf["close"] - cdf["open"])

cdf = cdf.reset_index()
del cdf['index']

for i in range(len(cdf)):
    tmp = cdf.loc[i,cdf.columns.values]
    if tmp['open'] <= tmp['close']:
        tmp['isGreen'] = 1
    else:
        tmp['isGreen'] = 0

    cdf.loc[i] = tmp.values

cdf

,open,close,high,low,volume,time,isGreen,bodyLen
0,65867.8,65856.8,66069.4,65685.9,2609.91,1711306800000,0.0,11.0
1,65354.0,65869.9,66088.0,65346.5,3323.76,1711303200000,1.0,515.9


In [37]:
cdf

,open,close,high,low,volume,time,isGreen,bodyLen
0,65867.8,65856.8,66069.4,65685.9,2609.91,1711306800000,0.0,11.0
1,65354.0,65869.9,66088.0,65346.5,3323.76,1711303200000,1.0,515.9


In [39]:
# for debug
    # threesoldiers
        # buy
# cdf.at[0,'isGreen'] = 1.0
# cdf.at[1,'isGreen'] = 1.0
# cdf.at[2,'isGreen'] = 1.0
# cdf.at[0,'bodyLen'] = 100
# cdf.at[1,'bodyLen'] = 10
# cdf.at[2,'bodyLen'] = 1
#         # sell
# cdf.at[0,'isGreen'] = 0.0
# cdf.at[1,'isGreen'] = 0.0
# cdf.at[2,'isGreen'] = 0.0
# cdf.at[0,'bodyLen'] = 100
# cdf.at[1,'bodyLen'] = 10
# cdf.at[2,'bodyLen'] = 1
#     # engulfing
#         # sell
# cdf.at[0,'isGreen'] = 0.0
# cdf.at[1,'isGreen'] = 1.0
# cdf.at[0,'bodyLen'] = 100
# cdf.at[1,'bodyLen'] = 10
#         # buy
# cdf.at[0,'isGreen'] = 1.0
# cdf.at[1,'isGreen'] = 0.0
# cdf.at[0,'bodyLen'] = 100
# cdf.at[1,'bodyLen'] = 10

cdf

,open,close,high,low,volume,time,isGreen,bodyLen
0,65867.8,65856.8,66069.4,65685.9,2609.91,1711306800000,1.0,100.0
1,65354.0,65869.9,66088.0,65346.5,3323.76,1711303200000,0.0,10.0


In [40]:
def checkPosition(strategy,interval):
    global threesoldiers_body_condition,engulfing_body_condition
    suitableForBuy = False
    suitableForSell = False
    
    if strategy == Strategy.THREESOLDIERS.value:

        if cdf.loc[0]['isGreen'] == 1.0:
            if cdf.loc[1]['isGreen'] == 1.0:
                if cdf.loc[2]['isGreen'] == 1.0:
                    if cdf.loc[0]['bodyLen'] / cdf.loc[1]['bodyLen'] >= threesoldiers_body_condition[1]:
                        if cdf.loc[1]['bodyLen'] / cdf.loc[2]['bodyLen'] >= threesoldiers_body_condition[0]:
                            suitableForBuy = True
                
        if cdf.loc[0]['isGreen'] == 0.0:
            if cdf.loc[1]['isGreen'] == 0.0:
                if cdf.loc[2]['isGreen'] == 0.0:
                    if cdf.loc[0]['bodyLen'] / cdf.loc[1]['bodyLen'] >= threesoldiers_body_condition[1]:
                        if cdf.loc[1]['bodyLen'] / cdf.loc[2]['bodyLen'] >= threesoldiers_body_condition[0]:
                            suitableForSell = True
                            
    if strategy == Strategy.ENGULFING.value:
        
        if cdf.loc[0]['isGreen'] == 0.0:
            if cdf.loc[1]['isGreen'] == 1.0 or engulfing_first_candle_color_dont_care:
                    if cdf.loc[0]['bodyLen'] / cdf.loc[1]['bodyLen'] >= engulfing_body_condition:
                        suitableForSell = True
                
        if cdf.loc[0]['isGreen'] == 1.0:
            if cdf.loc[1]['isGreen'] == 0.0 or engulfing_first_candle_color_dont_care:
                    if cdf.loc[0]['bodyLen'] / cdf.loc[1]['bodyLen'] >= engulfing_body_condition:
                        suitableForBuy = True 
    
    print(f'suitableForBuy: {suitableForBuy}')
    print(f'suitableForSell: {suitableForSell}')
    return suitableForBuy, suitableForSell


In [41]:
suitableForBuy = False
suitableForSell = False

suitableForBuy,suitableForSell = checkPosition(strategy.value,interval.value)

suitableForBuy: True
suitableForSell: False


In [42]:
def calculateAmount(total_balance,usdt,current_price,trade_percentage_current,learning_rate):
    amount_usdt = total_balance * trad_percentage_current / 100
    if amount_usdt > usdt:
        amount_usdt = usdt
    amount = amount_usdt / current_price
    trade_percentage_current *= learning_rate
    return amount

    
def buyBTC(bingx_client,pair,interval,total_balance,usdt_balance,buy_percentage_current,learning_rate):
  
    print("\033[92m \033[1m{}\033[00m" .format(f"buy opportunity!\n"))              
        
    if usdt_balance > 0:
        buyPositionsNumber += 1    
        
        res = bingx_client.perpetual_v2.market.get_k_line_data(pair,interval)
        last_price = float(res['close'])
        amountToBuy = calculateAmount(total_balance,usdt_balance,last_price,buy_percentage_current,learning_rate)
        
        print(f'current buy percentage: {buy_percentage_current}')
        order = types.Order(symbol=pair,side=types.Side.BUY,type=types.OrderType.MARKET,quote_order_qty=amountToBuy,quantity=amountToBuy)
        print(f'created order: {order.to_dict()}')
#         res = bingx_client.spot.trade.create_order(order)
        return res
        
    else:
        print(f"\tthere isn't any usdt to buy BTC with it!")
        return 0
            
def sellBTC(bingx_client,pair,interval,total_balance,bitcoin_balance,sell_percentage_current,learning_rate):
    
    print("\033[92m \033[1m{}\033[00m" .format(f"sell opportunity!\n"))      
        
    if bitcoin_balance > 0:
        sellPositionsNumber += 1    
        
        print(f'current sell percentage: {sell_percentage_current}')
        res = bingx_client.perpetual_v2.market.get_k_line_data(pair,interval)
        last_price = float(res['close'])
        currentValueOfAllBitcoins = bitcoin_balance * last_price
        amountToBuy = calculateAmount(total_balance,currentValueOfAllBitcoins,last_price,buy_percentage_current,learning_rate)

        order = types.Order(symbol=pair,side=types.Side.SELL,type=types.OrderType.MARKET,quote_order_qty=amountToSell,quantity=amountToSell)
        print(f'created order: {order.to_dict()}')
#         res = bingx_client.spot.trade.create_order(order)
        return res
        
    else:
        print(f"\tthere isn't any bitcoin to sell!")
        return 0
            

In [44]:
buy_percentage_init = 10
sell_percentage_init = 13
learning_rate = 1.3

buy_percentage_current = buy_percentage_init
sell_percentage_current = sell_percentage_init


if suitableForBuy:
    res = buyBTC(bingx_client,pair,interval,total_balance,usdt_balance,buy_percentage_current,learning_rate)
if suitableForSell:
    res = sellBTC(bingx_client,pair,interval,total_balance,bitcoin_balance,sell_percentage_current,learning_rate)
    
if suitableForBuy or suitableForSell:
    print(f'time: {time.ctime(time.time())}: res: {res}\n')

 buy opportunity!



UnboundLocalError: local variable 'buyPositionsNumber' referenced before assignment

## ###############

In [ ]:
def buyBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global buyDuplicateChecker,sellDuplicateChecker, learning_rate, buy_percentage_current
    global buy_percentage_init, sell_percentage_current, sell_percentage_init
   
    buyDuplicateChecker += 1
    sellDuplicateChecker = 0
    if strategy.name == Strategy.ENGULFING.name:
        if buyDuplicateChecker != 3:
            return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[92m \033[1m{}\033[00m" .format(f"buy opportunity!\n"))        

    if usdt_balance > 0:
        buyPositionsNumber += 1
        
        print(f'current buy percentage: {buy_percentage_current}')
        amountToBuy_usdt = total_balance * buy_percentage_current / 100
        buy_percentage_current *= learning_rate
        sell_percentage_current = sell_percentage_init
        
        if amountToBuy_usdt > usdt_balance:
            amountToBuy_usdt = usdt_balance
            
        addedBitcoinAmount = np.longdouble(amountToBuy_usdt / cdf.loc[i]["close"])
        usdt_balance -= amountToBuy_usdt
        bitcoin_balance += addedBitcoinAmount
        updateTotalBalance(i)
        print(f'\tbought {amountToBuy_usdt} $ BTC ({addedBitcoinAmount} BTCs) in {time.ctime(cdf.loc[i]["time"] // 1e3)}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')
        
    else:
        print(f"\tthere isn't any usdt to buy BTC with it!")
        print(f'\tcurrent BTC balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')

def sellBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global sellDuplicateChecker,buyDuplicateChecker, buy_percentage_current, buy_percentage_init
    global sell_percentage_current, sell_percentage_init, learning_rate
    
    sellDuplicateChecker += 1
    buyDuplicateChecker = 0
    
    if strategy.name == Strategy.ENGULFING.name:
        if sellDuplicateChecker != 3:
            return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[91m \033[1m{}\033[00m" .format(f"sell opportunity!\n"))        
    if bitcoin_balance > 0:
        sellPositionsNumber += 1
        
        currentValueOfAllBitcoins = bitcoin_balance * cdf.loc[i]["close"]
        
        print(f'current sell percentage: {sell_percentage_current}')
        amountToSell_usdt = total_balance * sell_percentage_current / 100
        sell_percentage_current *= learning_rate
        buy_percentage_current = buy_percentage_init
        
        if amountToSell_usdt > currentValueOfAllBitcoins:
            amountToSell_usdt = currentValueOfAllBitcoins
        
        amountToSell_bitcoin = amountToSell_usdt / cdf.loc[i]["close"]
        usdt_balance += amountToSell_usdt
        bitcoin_balance -= amountToSell_bitcoin
        updateTotalBalance(i)
        print(f'\tsold {amountToSell_usdt} $ of BTC ({amountToSell_usdt} $) in {time.ctime(cdf.loc[i]["time"] // 1e3)}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')
        
    else:
        print(f"\tthere isn't any bitcoin to sell!")
        print(f'\tcurrent USDT balance: {usdt_balance} $')



In [129]:
from bingX.spot import types

In [153]:
temp = types.Order(symbol='TRX-USDT',side=types.Side.SELL,type=types.OrderType.MARKET,quote_order_qty=49.95,quantity=49.95)

In [154]:
temp.to_dict()

{'symbol': 'TRX-USDT',
 'side': 'SELL',
 'type': 'MARKET',
 'quantity': 49.95,
 'quote_order_qty': 49.95}

In [155]:
res = bingx_client.spot.trade.create_order(temp)
res

{'symbol': 'TRX-USDT',
 'orderId': 1771627059596918784,
 'transactTime': 1711223780903,
 'price': '0.11931',
 'stopPrice': '0',
 'origQty': '49.9',
 'executedQty': '49.9',
 'cummulativeQuoteQty': '5.954142918983257',
 'status': 'FILLED',
 'type': 'MARKET',
 'side': 'SELL',
 'clientOrderID': ''}

In [170]:
if interval.value == '1h':
    timeOffset = 60 * 60 * 1000
if interval.value == '4h':
    timeOffset = 4 * 60 * 60 * 1000
if interval.value == '1d':
    timeOffset = 24 * 60 * 60 * 1000
if interval.value == '1m':
    timeOffset = 60 * 1000
    
start_time = round(time.time() * 1e3) - timeOffset * (strategy.value)
end_time = round(time.time() * 1e3)
print(f'difference(ms): {end_time - start_time}')
print(f'difference(hour): {(end_time - start_time) / 1e3 / 60 / 60}')

difference(ms): 180000
difference(hour): 0.05


In [171]:
bingx_client = BingX(api_key=api_key, secret_key=secret_key)

In [172]:
candleData = bingx_client.perpetual_v2.market.get_k_line_data(pair,interval.value,start_time,end_time)
candleData # last index is the last candle and so forth

[{'open': '64591.3',
  'close': '64599.3',
  'high': '64621.9',
  'low': '64570.8',
  'volume': '26.14',
  'time': 1711199220000},
 {'open': '64598.6',
  'close': '64571.0',
  'high': '64633.5',
  'low': '64565.4',
  'volume': '53.55',
  'time': 1711199280000}]

In [173]:
import warnings
warnings.filterwarnings('ignore')

df = pd.DataFrame(candleData)
cdf = df.iloc[::-1]

cdf['open'] = pd.to_numeric(cdf['open'])
cdf['high'] = pd.to_numeric(cdf['high'])
cdf['low'] = pd.to_numeric(cdf['low'])
cdf['close'] = pd.to_numeric(cdf['close'])

cdf['isGreen'] = cdf["close"]
cdf['bodyLen'] = abs(cdf["close"] - cdf["open"])

cdf = cdf.reset_index()
del cdf['index']

for i in range(len(cdf)):
    tmp = cdf.loc[i,cdf.columns.values]
    if tmp['open'] <= tmp['close']:
        tmp['isGreen'] = 1
    else:
        tmp['isGreen'] = 0

    cdf.loc[i] = tmp.values

cdf

,open,close,high,low,volume,time,isGreen,bodyLen
0,64598.6,64571.0,64633.5,64565.4,53.55,1711199280000,0.0,27.6
1,64591.3,64599.3,64621.9,64570.8,26.14,1711199220000,1.0,8.0


In [174]:
cdf.describe()

,open,close,high,low,time,isGreen,bodyLen
count,2.00000,2.000000,2.000000,2.000000,2.000000e+00,2.000000,2.000000
mean,64594.95000,64585.150000,64627.700000,64568.100000,1.711199e+12,0.500000,17.800000
std,5.16188,20.011122,8.202439,3.818377,4.242641e+04,0.707107,13.859293
min,64591.30000,64571.000000,64621.900000,64565.400000,1.711199e+12,0.000000,8.000000
25%,64593.12500,64578.075000,64624.800000,64566.750000,1.711199e+12,0.250000,12.900000
50%,64594.95000,64585.150000,64627.700000,64568.100000,1.711199e+12,0.500000,17.800000
75%,64596.77500,64592.225000,64630.600000,64569.450000,1.711199e+12,0.750000,22.700000
max,64598.60000,64599.300000,64633.500000,64570.800000,1.711199e+12,1.000000,27.600000


## dataset is ready

In [157]:
res = bingx_client.spot.trade.get_assets()
res

{'balances': [{'asset': 'BTC', 'free': '0.00041958', 'locked': '0'},
  {'asset': 'USDT', 'free': '62.1485304608547', 'locked': '20.718'},
  {'asset': 'TRX', 'free': '0.05', 'locked': '0'},
  {'asset': 'VST', 'free': '0', 'locked': '0'},
  {'asset': 'XRP', 'free': '0', 'locked': '0'}]}

In [139]:
for record in res['balances']:
    if record['asset'] == 'USDT':
        usdt_balance = float(record['free'])
    if record['asset'] == 'BTC':
        bitcoin_balance = float(record['free'])

print(f'usdt balance: {usdt_balance}\n')
print(f'bitcoin balance: {bitcoin_balance}')

usdt balance: 62.17024165212848

bitcoin balance: 0.00041958


In [177]:
calculatePeriod()

In [178]:
period

0

In [179]:
totals = []
counter = 0
total_counter = 0

calculatePeriod()
for i in range(len(cdf) -1, -1, -1):
    checkForPosition(i,strategy)
    counter += 1
    if counter == period:
        totals.append(timeout(i))

i = 0:
